<a href="https://colab.research.google.com/github/stanley-zhou/CIS5500_Final_Project_Group15/blob/main/Data_Preprocessing_and_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
BASE_DIR = f"/content/drive/Shareddrives/CIS5500/IMDb_Datasets"
BASICS_PATH  = f"{BASE_DIR}/title.basics.tsv"
RATINGS_PATH = f"{BASE_DIR}/title.ratings.tsv"
AKAS_PATH    = f"{BASE_DIR}/title.akas.tsv"
CREW_PATH    = f"{BASE_DIR}/title.crew.tsv"

In [3]:
!pip install pycountry

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pycountry

### **Load, Preprocess, and Clean `basics` dataset**

In [4]:
basics_df = pd.read_csv(BASICS_PATH, sep='\t', low_memory=True)

In [5]:
basics_df.head(5)

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000001,short,Carmencita,Carmencita,0,1894,\N,1,"Documentary,Short"
1,tt0000002,short,Le clown et ses chiens,Le clown et ses chiens,0,1892,\N,5,"Animation,Short"
2,tt0000003,short,Poor Pierrot,Pauvre Pierrot,0,1892,\N,5,"Animation,Comedy,Romance"
3,tt0000004,short,Un bon bock,Un bon bock,0,1892,\N,12,"Animation,Short"
4,tt0000005,short,Blacksmith Scene,Blacksmith Scene,0,1893,\N,1,Short


In [6]:
basics_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12029264 entries, 0 to 12029263
Data columns (total 9 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   tconst          object
 1   titleType       object
 2   primaryTitle    object
 3   originalTitle   object
 4   isAdult         int64 
 5   startYear       object
 6   endYear         object
 7   runtimeMinutes  object
 8   genres          object
dtypes: int64(1), object(8)
memory usage: 826.0+ MB


Check null values for each features

In [7]:
basics_df.isnull().sum()

,0
tconst,0
titleType,0
primaryTitle,22
originalTitle,22
isAdult,0
startYear,0
endYear,0
runtimeMinutes,0
genres,610


For our implementation, movie titles and genres are necessary. So we dropped the rows where `primaryTitle`, `originalTitle`, `genres` are null.

In [5]:
basics_df = basics_df.dropna(subset=['primaryTitle', 'originalTitle', 'genres'])

Drop unrelated columns and Rename some columns

In [6]:
basics_df = basics_df.drop(columns=['originalTitle', 'endYear'])
basics_df = basics_df.rename(columns={'primaryTitle': 'title', 'tconst': 'movieId'})

Keep only rows where `titleType == "movie"` and rename `titleType` as `movieType`

In [7]:
basics_df = basics_df[basics_df['titleType'] == 'movie']
basics_df = basics_df.rename(columns={'titleType': 'movieType'})

Handle `genres` Column

In [8]:
genre_exploded = basics_df.copy()
genre_exploded['genres'] = genre_exploded['genres'].str.split(',')
genre_exploded = genre_exploded.explode('genres', ignore_index=True)
genre_exploded = genre_exploded[genre_exploded['genres'] != '\\N']
genre_exploded = genre_exploded.rename(columns={'genres': 'genre'}).reset_index(drop=True)

In [10]:
movie_ids = set(basics_df["movieId"].unique())

### **Load, Preprocess, and Clean `rating` dataset**

In [11]:
ratings_df = pd.read_csv(RATINGS_PATH, sep='\t', na_values = '\\N', low_memory=True)

In [31]:
ratings_df.head()

,tconst,averageRating,numVotes
0,tt0000001,5.7,2183
1,tt0000002,5.5,302
2,tt0000003,6.4,2263
3,tt0000004,5.2,194
4,tt0000005,6.2,3001


In [32]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1632106 entries, 0 to 1632105
Data columns (total 3 columns):
 #   Column         Non-Null Count    Dtype  
---  ------         --------------    -----  
 0   tconst         1632106 non-null  object 
 1   averageRating  1632106 non-null  float64
 2   numVotes       1632106 non-null  int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 37.4+ MB


In [33]:
ratings_df.isnull().sum()

,0
tconst,0
averageRating,0
numVotes,0


Rename columns to match

In [12]:
ratings_df = ratings_df.rename(columns={'tconst': 'movieId', 'averageRating': 'rating'})

Keep only movies

In [13]:
ratings_df = ratings_df[ratings_df['movieId'].isin(movie_ids)].copy()

### **Load, Preprocess, and Clean `akas` dataset**

In [15]:
akas_df = pd.read_csv(AKAS_PATH, sep='\t', usecols = ['region', 'title', 'isOriginalTitle'], low_memory=True)

Filter only rows where `isOriginalTitle` = 1

In [16]:
akas_df = akas_df[akas_df['isOriginalTitle'] == 1]

In [17]:
akas_df.head(10)

,title,region,isOriginalTitle
0,Carmencita,\N,1
8,Le clown et ses chiens,\N,1
16,Pauvre Pierrot,\N,1
27,Un bon bock,\N,1
35,Blacksmith Scene,\N,1
47,Chinese Opium Den,\N,1
53,Corbett and Courtney Before the Kinetograph,\N,1
64,Edison Kinetoscopic Record of a Sneeze,\N,1
75,Miss Jerry,\N,1
81,La sortie de l'usine Lumière à Lyon,\N,1


In [20]:
akas_df['region'].unique()

array(['\\N'], dtype=object)

Original plan: From akads_df, we choose the rows with `isOriginalTitle` = 1 and use the corresponding `region` value to infer where the movei is made.

However, notice that whether `isOriginalTitle` = 1, `region` has null value. This might be because this dataset is specifically designed to show the althernative names of the movie and their corresponding regions. So it won't show the reigion that uses original names (which is highly likely to be the origin country of the movie). Now this becomes a problem here: how to get the origin country of the movie.

Potential ways:
- Find datasets and merge by movie name
- Create functions to do web searching (recall CIS5450 ceo datasets)
- Any better ideas?

### **Load, Preprocess, and Clean `crew` dataset**

In [14]:
crew_df = pd.read_csv(CREW_PATH, sep='\t', na_values = '\\N', low_memory=True)

In [37]:
crew_df.head()

,tconst,directors,writers
0,tt0000001,nm0005690,NaN
1,tt0000002,nm0721526,NaN
2,tt0000003,nm0721526,nm0721526
3,tt0000004,nm0721526,NaN
4,tt0000005,nm0005690,NaN


In [38]:
crew_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12027384 entries, 0 to 12027383
Data columns (total 3 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   tconst     object
 1   directors  object
 2   writers    object
dtypes: object(3)
memory usage: 275.3+ MB


In [39]:
crew_df.isnull().sum()

,0
tconst,0
directors,5256887
writers,5940576


Rename columns to match

In [15]:
crew_df = crew_df.rename(columns={'tconst': 'movieId'})

Keep only movies

In [16]:
crew_df = crew_df[crew_df['movieId'].isin(movie_ids)].copy()

Separate to two tables for directors and writers

In [21]:
# Define a helper function that turns a cell into a list
def split_to_list(cell):
    """
    If cell is NaN -> return empty list [].
    If cell is a string like 'nm0001,nm0002' -> ['nm0001', 'nm0002'].
    """
    if pd.isna(cell):
        return []
    # Ensure it's a string, split on comma, and strip spaces (just in case)
    return [x.strip() for x in str(cell).split(",") if x.strip() != ""]

# Apply to directors and writers to create list columns
crew_df["directors_list"] = crew_df["directors"].apply(split_to_list)
crew_df["writers_list"]   = crew_df["writers"].apply(split_to_list)

# Quick check
crew_df[["movieId", "directors", "directors_list", "writers", "writers_list"]].head()


,movieId,directors,directors_list,writers,writers_list
8,tt0000009,nm0085156,[nm0085156],nm0085156,[nm0085156]
144,tt0000147,nm0714557,[nm0714557],NaN,[]
331,tt0000335,"nm0095714,nm0675140","[nm0095714, nm0675140]",NaN,[]
498,tt0000502,nm0063413,[nm0063413],"nm0063413,nm0657268,nm0675388","[nm0063413, nm0657268, nm0675388]"
570,tt0000574,nm0846879,[nm0846879],nm0846879,[nm0846879]


In [23]:
# Create numeric count columns for number of directors and writers

# Number of directors = length of directors_list
crew_df["num_directors"] = crew_df["directors_list"].str.len()

# Number of writers = length of writers_list
crew_df["num_writers"] = crew_df["writers_list"].str.len()

# Sanity check: look at basic stats
print("num_directors summary:")
print(crew_df["num_directors"].describe())

print("\nnum_writers summary:")
print(crew_df["num_writers"].describe())

# Peek at a few rows
crew_df[["movieId", "directors_list", "num_directors",
         "writers_list", "num_writers"]].head()


num_directors summary:
count    730269.000000
mean          1.005351
std           0.722715
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          85.000000
Name: num_directors, dtype: float64

num_writers summary:
count    730269.000000
mean          1.286942
std           1.219724
min           0.000000
25%           1.000000
50%           1.000000
75%           2.000000
max          69.000000
Name: num_writers, dtype: float64


,movieId,directors_list,num_directors,writers_list,num_writers
8,tt0000009,[nm0085156],1,[nm0085156],1
144,tt0000147,[nm0714557],1,[],0
331,tt0000335,"[nm0095714, nm0675140]",2,[],0
498,tt0000502,[nm0063413],1,"[nm0063413, nm0657268, nm0675388]",3
570,tt0000574,[nm0846879],1,[nm0846879],1


In [25]:
# Explode directors_list so each (movieID, director) is its own row

# Start from only the columns we need
directors_exploded_df = crew_df[["movieId", "directors_list"]].copy()

# Explode the list column
# Example:
# movieID = tt123, directors_list = ["nm1", "nm2"]
# ->
# tt123, "nm1"
# tt123, "nm2"
directors_exploded_df = directors_exploded_df.explode("directors_list", ignore_index=True)

# Drop rows where directors_list is empty/NaN after exploding
directors_exploded_df = directors_exploded_df[
    directors_exploded_df["directors_list"].notna() &
    (directors_exploded_df["directors_list"] != "")
].copy()

# Rename column to something cleaner
directors_exploded_df = directors_exploded_df.rename(
    columns={"directors_list": "director_id"}
)

# Remove duplicates just in case (same director listed twice for same movie)
directors_exploded_df = directors_exploded_df.drop_duplicates()

print("directors_exploded_df shape:", directors_exploded_df.shape)
directors_exploded_df.head()


directors_exploded_df shape: (734177, 2)


,movieId,director_id
0,tt0000009,nm0085156
1,tt0000147,nm0714557
2,tt0000335,nm0095714
3,tt0000335,nm0675140
4,tt0000502,nm0063413


In [26]:
# Explode writers_list so each (movieID, writer) is its own row

# Start from only the columns we need
writers_exploded_df = crew_df[["movieId", "writers_list"]].copy()

# Explode the list column
writers_exploded_df = writers_exploded_df.explode("writers_list", ignore_index=True)

# Drop rows where writers_list is empty/NaN after exploding
writers_exploded_df = writers_exploded_df[
    writers_exploded_df["writers_list"].notna() &
    (writers_exploded_df["writers_list"] != "")
].copy()

# Rename column
writers_exploded_df = writers_exploded_df.rename(
    columns={"writers_list": "writer_id"}
)

# Remove duplicates
writers_exploded_df = writers_exploded_df.drop_duplicates()

print("writers_exploded_df shape:", writers_exploded_df.shape)
writers_exploded_df.head()


writers_exploded_df shape: (939814, 2)


,movieId,writer_id
0,tt0000009,nm0085156
3,tt0000502,nm0063413
4,tt0000502,nm0657268
5,tt0000502,nm0675388
6,tt0000574,nm0846879
